<a href="https://colab.research.google.com/github/aimldstejas/aibits-genai-notebooks/blob/main/course-3-ai-agents/lab-06-research-assistant-for-fernwood.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Lab 6 (graded) — Research assistant for Fernwood
**Course 3: AI Agents and Agentic AI with Python — Chapter 6: Agentic retrieval & research**

**Problem brief (Sam Okafor, Fernwood Media):** "Reporters spend hours gathering
background. Build an assistant that researches a topic across sources and hands back a
cited brief — real iterative digging, not a single lookup."

**What you'll submit:** an iterative retrieval loop hitting an answer-correctness + citation-
accuracy target on multi-hop questions, and a demonstrated budget-respecting stop condition.

In [ ]:
!pip install -q sentence-transformers ddgs

## 1. A real internal corpus, deliberately split across documents
so that answering the multi-hop question below genuinely requires TWO retrieval hops — a
single lookup can't find the answer.

In [ ]:
corpus = [
    {'id': 'doc1', 'title': 'Fernwood Media 2024 overview',
     'text': 'Fernwood Media\'s technology desk is led by senior editor Priya Nair, who joined the outlet in 2019.'},
    {'id': 'doc2', 'title': 'Fernwood staff bios',
     'text': 'Priya Nair previously worked as a data engineer at a logistics company before moving into journalism.'},
    {'id': 'doc3', 'title': 'Fernwood editorial calendar',
     'text': 'The business desk covers quarterly earnings season with a dedicated team of three reporters.'},
    {'id': 'doc4', 'title': 'Fernwood sports desk',
     'text': 'The sports desk expanded coverage of regional leagues in 2023.'},
]

multi_hop_question = {
    'question': "What was the technology desk editor's previous job before journalism?",
    'answer_contains': 'data engineer',
    'required_docs': {'doc1', 'doc2'},  # doc1 names the editor; doc2 gives their prior job
}

## 2. Retrieval + a (free, no-key) web search tool

In [ ]:
from sentence_transformers import SentenceTransformer
import numpy as np

embedder = SentenceTransformer('all-MiniLM-L6-v2')
corpus_embeddings = embedder.encode([d['text'] for d in corpus], normalize_embeddings=True)

def retrieve_corpus(query, top_k=1):
    q = embedder.encode([query], normalize_embeddings=True)
    sims = (corpus_embeddings @ q.T).ravel()
    idx = np.argsort(-sims)[:top_k]
    return [{'id': corpus[i]['id'], 'title': corpus[i]['title'], 'text': corpus[i]['text'], 'score': float(sims[i])} for i in idx]

def web_search(query, max_results=2):
    try:
        from ddgs import DDGS
        with DDGS() as ddgs:
            results = list(ddgs.text(query, max_results=max_results))
        return [{'title': r['title'], 'text': r['body'], 'id': r['href']} for r in results]
    except Exception as e:
        print(f'  (live web search unavailable: {e} — using cached results)')
        return [{'title': 'cached result', 'text': f'No live web access; this is a placeholder for: {query}', 'id': 'cache'}]

## 3. The iterative retrieve-read-refine loop

In [ ]:
def research(question, max_hops=3, cost_budget=5):
    notes = []      # (fact, source_id) — carries citations forward
    sources_used = set()
    cost_spent = 0
    current_query = question

    for hop in range(max_hops):
        if cost_spent >= cost_budget:
            return {'status': 'budget_exceeded', 'notes': notes, 'sources': sources_used}

        results = retrieve_corpus(current_query, top_k=1)
        cost_spent += 1
        for r in results:
            if r['id'] not in sources_used:
                notes.append((r['text'], r['id']))
                sources_used.add(r['id'])

        # "do I have enough?" heuristic: if the question asks about a PERSON's prior role and
        # we've only found the person's NAME so far (not a job description), do another hop
        # keyed on that name. A real system would ask an LLM this question directly.
        latest_text = notes[-1][0] if notes else ''
        if 'previous' in question.lower() and 'editor' in latest_text.lower() and 'led by' in latest_text.lower():
            name = latest_text.split('led by')[-1].split(',')[0].strip()
            current_query = f'{name} previous job before journalism'
        else:
            break  # enough evidence gathered, or no more refinement heuristic applies

    return {'status': 'done', 'notes': notes, 'sources': sources_used}

result = research(multi_hop_question['question'])
print('Status:', result['status'])
print('Sources used:', result['sources'])
for text, src in result['notes']:
    print(f'  [{src}] {text}')

## 4. Evaluate: answer correctness + citation accuracy

In [ ]:
combined_notes_text = ' '.join(t for t, _ in result['notes']).lower()
answer_correct = multi_hop_question['answer_contains'] in combined_notes_text
citations_correct = multi_hop_question['required_docs'] <= result['sources']

print(f'Answer correctness: {answer_correct}  (looking for "{multi_hop_question["answer_contains"]}")')
print(f'Citation accuracy:  {citations_correct}  (required {multi_hop_question["required_docs"]}, got {result["sources"]})')
assert answer_correct and citations_correct, 'The iterative loop should have chained doc1 -> doc2 to find the answer.'
print('\nBoth targets hit — the single-hop retriever alone (top_k=1 on the raw question) would')
print('have stopped at doc1 and missed the actual answer; confirm this below.')

In [ ]:
single_hop = retrieve_corpus(multi_hop_question['question'], top_k=1)
single_hop_correct = multi_hop_question['answer_contains'] in single_hop[0]['text'].lower()
print(f'Single-shot retrieval alone finds the answer: {single_hop_correct} (expected: False)')
assert not single_hop_correct, 'Single-shot retrieval should NOT find this multi-hop answer.'

## 5. Stop condition respecting a budget

In [ ]:
tight_result = research(multi_hop_question['question'], cost_budget=1)
print('With a budget of 1:', tight_result['status'], '- sources:', tight_result['sources'])
print('The loop correctly stops rather than running unbounded, at the cost of an incomplete answer.')

## 6. Write-up (fill in)
The "do I have enough?" heuristic above is hand-coded for this specific question shape. What
would break it on a differently-phrased multi-hop question, and what would a more general
version look like (hint: Chapter 1's agent loop already has the right building block)?

_Your answer here._

---
*Beacon AI · AIBits Academy — Chapter 6: Agentic retrieval & research*